<a href="https://colab.research.google.com/github/kurkur19/NLP_al_khmuz/blob/main/Al_Khmuz_NLP_5lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Аль Хмуз Карина Бассамівна БС-25
Практична робота №5



**РІШЕННЯ ЗАДАЧІ POS-ТЕГУВАННЯ ЗА ДОПОМОГОЮ ЗГОРТКОВИХ НЕЙРОННИХ МЕРЕЖ**

Встановлюємо бібліотеку pyconll, яка використовується для роботи з корпусами в форматі CoNLL (формат для лінгвістичних даних).

In [1]:
!pip install pyconll

Імпортуємо бібліотеки

In [2]:
import os
import requests
import torch
from torch.utils.data import TensorDataset
import pyconll
import os
import requests
from collections import Counter

Також оголошено словник DATASETS, який містить посилання на датасети для навчання, валідації та тестування.

In [3]:
DATASETS = {
    "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Ukrainian-IU/master/uk_iu-ud-train.conllu",
    "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Ukrainian-IU/master/uk_iu-ud-dev.conllu",
    "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Ukrainian-IU/master/uk_iu-ud-test.conllu"
}

import os

def save_model(model, filepath):
    """Збереження моделі в файл"""
    torch.save(model.state_dict(), filepath)
    print(f"Модель збережено в {filepath}")

def load_model(model, filepath, device="cpu"):
    """Завантаження моделі з файлу"""
    model.load_state_dict(torch.load(filepath, map_location=device))
    model.to(device)
    print(f"Модель завантажено з {filepath}")

# Функція завантаження
def download_corpus(url, save_path):
    if not os.path.exists(save_path):
        print(f"Завантаження {save_path}...")
        response = requests.get(url)
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"{save_path} завантажено.")
    else:
        print(f"{save_path} вже завантажено.")
    return save_path

Завантажуємо дані з вказаних URL для кожного набору (train, dev, test) і зберігаємо їх у файли. Потім завантажуємо ці файли за допомогою бібліотеки pyconll і підраховуємо кількість речень у кожному наборі:
•	Виведено кількість речень у наборах для навчання, валідації та тестування.


In [4]:
corpora_paths = {}
for split, url in DATASETS.items():
    save_path = f"ukrainian_pos_{split}.conllu"
    corpora_paths[split] = download_corpus(url, save_path)

train_data = pyconll.load_from_file(corpora_paths["train"])
dev_data = pyconll.load_from_file(corpora_paths["dev"])
test_data = pyconll.load_from_file(corpora_paths["test"])

print(f"Train: {len(train_data)} речень")
print(f"Dev: {len(dev_data)} речень")
print(f"Test: {len(test_data)} речень")

Завантаження ukrainian_pos_train.conllu...
ukrainian_pos_train.conllu завантажено.
Завантаження ukrainian_pos_dev.conllu...
ukrainian_pos_dev.conllu завантажено.
Завантаження ukrainian_pos_test.conllu...
ukrainian_pos_test.conllu завантажено.
Train: 5521 речень
Dev: 673 речень
Test: 898 речень


Я вивела перші два речення з навчального набору. Для кожного речення виведено форму слова (token.form) та його універсальну частину мови (token.upos). Це дозволяє побачити структуру та лінгвістичні характеристики кожного токена в реченнях.

In [5]:
for i, sentence in enumerate(train_data[:2]):
    print(f"Речення {i+1}:")
    for token in sentence:
        print(f"{token.form}\t{token.upos}")
    print("\n")

Речення 1:
У	ADP
домі	NOUN
римського	ADJ
патриція	NOUN
Руфіна	PROPN
була	VERB
прегарна	ADJ
фреска	NOUN
,	PUNCT
зображення	NOUN
Венери	PROPN
та	CCONJ
Адоніса	PROPN
.	PUNCT


Речення 2:
Якось	ADV
зібралися	VERB
у	ADP
нього	PRON
,	PUNCT
ховаючися	VERB
від	ADP
переслідувань	NOUN
,	PUNCT
одновірці	NOUN
дружини	NOUN
–	PUNCT
християнки	NOUN
.	PUNCT




Я знайшла максимальну довжину речення та максимальну довжину токена в навчальному наборі:
•	max_sentence_length — максимальна кількість токенів у реченні.
•	max_token_length — максимальна кількість символів у токені.
Це допомагає зрозуміти, які найбільші значення по довжині є в даних для подальшої обробки.


In [6]:
max_sentence_length = max(len(sentence) for sentence in train_data)
max_token_length = max(len(token.form) for sentence in train_data for token in sentence)
print(f"Максимальна довжина речення: {max_sentence_length}")
print(f"Максимальна довжина токена: {max_token_length}")

Максимальна довжина речення: 197
Максимальна довжина токена: 115


Я створила функцію build_char_vocab, яка будує словник для символів з тексту:
1.	char_counter підраховує частоти символів у всіх токенах.
2.	Створюється словник char_vocab, де кожен символ отримує унікальний індекс.
3.	В словнику також є спеціальний символ <PAD> з індексом 0 для заповнення.
Ця функція допомагає створити словник символів для подальшої обробки текстових даних.


In [7]:
def build_char_vocab(data):
    char_counter = Counter()
    for sentence in data:
        for token in sentence:
            char_counter.update(token.form)
    char_vocab = {"<PAD>": 0, **{char: idx + 1 for idx, char in
    enumerate(sorted(char_counter.keys()))}}
    return char_vocab



Я створила словник символів за допомогою функції build_char_vocab на основі навчальних даних. Виведено розмір цього словника, що показує кількість унікальних символів, які будуть використовуватися в подальшій обробці тексту.

In [8]:
char_vocab = build_char_vocab(train_data)
print(f"Розмір словника символів: {len(char_vocab)}")

Розмір словника символів: 190


Я вивела перші 10 символів зі словника символів, показавши, як кожен символ відповідає своєму унікальному індексу в словнику. Це дозволяє побачити, як побудований словник для подальшої роботи з текстом.

In [9]:
print("Перші 10 символів у словнику:")
for char, idx in list(char_vocab.items())[:10]:
    print(f"Символ: '{char}' -> Номер: {idx}")


Перші 10 символів у словнику:
Символ: '<PAD>' -> Номер: 0
Символ: ' ' -> Номер: 1
Символ: '!' -> Номер: 2
Символ: '"' -> Номер: 3
Символ: '#' -> Номер: 4
Символ: '$' -> Номер: 5
Символ: '%' -> Номер: 6
Символ: '&' -> Номер: 7
Символ: ''' -> Номер: 8
Символ: '(' -> Номер: 9


Я підрахувала частоти символів у всіх токенах навчального набору за допомогою Counter. Виведено 10 найчастотніших символів з їх відповідними частотами. Це дає уявлення про те, які символи зустрічаються найчастіше в даних.

In [10]:
from collections import Counter
char_counter = Counter()
for sentence in train_data:
    for token in sentence:
        char_counter.update(token.form)
print("\nНайчастотніші символи:")
for char, count in char_counter.most_common(10):
    print(f"Символ: '{char}' -> Частота: {count}")


Найчастотніші символи:
Символ: 'о' -> Частота: 38005
Символ: 'а' -> Частота: 33855
Символ: 'н' -> Частота: 27248
Символ: 'и' -> Частота: 26678
Символ: 'і' -> Частота: 23632
Символ: 'в' -> Частота: 20986
Символ: 'т' -> Частота: 20490
Символ: 'е' -> Частота: 18769
Символ: 'р' -> Частота: 18506
Символ: 'с' -> Частота: 16068


Я створила функцію build_pos_vocab, яка будує словник для універсальних частин мови (POS) з даних

In [11]:
def build_pos_vocab(data):
    pos_tags = sorted({token.upos for sentence in data for token in sentence
                       if token.upos is not None})
    pos_vocab = {pos: idx + 1 for idx, pos in enumerate(pos_tags)}
    pos_vocab["<PAD>"] = 0
    return pos_vocab

Я створила словник частин мови pos_vocab на основі навчальних даних і вивів його розмір, щоб побачити кількість унікальних частин мови в даних. Також додано перевірку, щоб переконатися, що в словнику немає значення None, яке могло б виникнути через відсутність POS-тегів у деяких токенах.

In [12]:
pos_vocab = build_pos_vocab(train_data)
print(f"Розмір словника частин мови: {len(pos_vocab)}")
assert None not in pos_vocab

Розмір словника частин мови: 18


Я створила функцію get_pos_examples, яка знаходить приклади для кожної частини мови з даних:
1.	Функція створює словник pos_examples, де для кожного POS-тегу (окрім <PAD>) зберігаються приклади токенів.
2.	Для кожного токена в даних, якщо його POS-тег ще не зібрав достатньо прикладів (за умовою num_examples), додано його форму до списку відповідного POS-тегу.
Це дозволяє отримати приклади для кожної частини мови в даному наборі.


In [13]:
def get_pos_examples(data, pos_vocab, num_examples=3):
    pos_examples = {pos: [] for pos in pos_vocab.keys() if pos != "<PAD>"}
    for sentence in data:
        for token in sentence:
            pos = token.upos
            if pos in pos_examples and len(pos_examples[pos]) < num_examples:
                pos_examples[pos].append(token.form)
    return pos_examples

Я створила функцію convert_to_tensors, яка перетворює дані у тензори:
1.	char_tensor — тензор для символів, де кожен токен представлений індексами символів (з обмеженням на максимальну довжину токена).
2.	pos_tensor — тензор для частин мови, де кожен токен має індекс відповідного POS-тегу.
3.	Для кожного речення в даних, функція заповнює ці тензори символами та частинами мови відповідно до заданих обмежень на довжину токенів і речень.
Це дозволяє підготувати дані для подальшої обробки в нейронній мережі.


In [14]:
def convert_to_tensors(data, char_vocab, pos_vocab, max_token_len, max_sent_len):
    num_sentences = len(data)
    char_tensor = torch.zeros((num_sentences, max_sent_len, max_token_len), dtype=torch.long)
    pos_tensor = torch.zeros((num_sentences, max_sent_len), dtype=torch.long)

    for i, sentence in enumerate(data):
        for j, token in enumerate(sentence):
            if j >= max_sent_len:
                break

            token_chars = [char_vocab.get(char, 0) for char in token.form[:max_token_len]]
            char_tensor[i, j, :len(token_chars)] = torch.tensor(token_chars)
            pos_tensor[i, j] = pos_vocab.get(token.upos, 0)

    return char_tensor, pos_tensor

Я обчислила максимальну довжину токенів і речень у навчальних даних, а потім перетворив ці дані у тензори для символів та POS-тегів:
1.	max_token_len — максимальна довжина токена.
2.	max_sent_len — максимальна кількість токенів у реченні.
3.	За допомогою функції convert_to_tensors я створила тензори для символів і POS-тегів для навчального, валідаційного та тестового наборів.
Виведено форму тензорів для символів та POS-тегів для навчального набору, що показує їх розміри.


In [15]:
max_token_len = max(len(token.form) for sentence in train_data for token in sentence)
max_sent_len = max(len(sentence) for sentence in train_data)

train_char_tensor, train_pos_tensor = convert_to_tensors(train_data,
    char_vocab, pos_vocab, max_token_len, max_sent_len)
dev_char_tensor, dev_pos_tensor = convert_to_tensors(dev_data,
    char_vocab, pos_vocab, max_token_len, max_sent_len)
test_char_tensor, test_pos_tensor = convert_to_tensors(test_data,
    char_vocab, pos_vocab, max_token_len, max_sent_len)

print(f"Форма тензора символів: {train_char_tensor.shape}")
print(f"Форма тензора POS-тегів: {train_pos_tensor.shape}")

Форма тензора символів: torch.Size([5521, 197, 115])
Форма тензора POS-тегів: torch.Size([5521, 197])


Я створила функцію add_special_columns, яка додає спеціальні стовпці до тензора символів:
1.	Для кожного токена додаються два додаткових стовпці — один на початку та один в кінці.
2.	Це робиться для того, щоб можна було враховувати спеціальні символи, наприклад, для вирішення задачі з обробки текстів (наприклад, для початку та кінця токенів).
Функція повертає тензор з додатковими стовпцями для кожного символу.


In [16]:
def add_special_columns(char_tensor):
    batch_size, max_sent_len, max_token_len = char_tensor.shape
    special_tensor = torch.zeros((batch_size, max_sent_len, max_token_len + 2),
                                 dtype=torch.long)
    special_tensor[:, :, 1:-1] = char_tensor
    return special_tensor

In [17]:
train_char_tensor = add_special_columns(train_char_tensor)
dev_char_tensor = add_special_columns(dev_char_tensor)
test_char_tensor = add_special_columns(test_char_tensor)

print(train_char_tensor[1][:5])
print(train_pos_tensor[1])
print(f"Оновлена форма тензора символів: {train_char_tensor.shape}")

tensor([[  0, 139, 150, 154, 157, 167,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0],
        [  0, 147, 171, 141, 156, 140, 151, 148, 157, 169,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0

Я створила дві моделі:
1.	ConvBlock — базовий блок з одною згортковою операцією (Conv1d), дроп-аутом та активацією LeakyReLU. Він виконує операцію згортки з заданим ядром та доповнює результат активацією й дроп-аутом.
2.	ConvResNet — мережа, яка складається з декількох ConvBlock. Кожен блок додається до попереднього результату (механізм резидуальних зв'язків), що допомагає уникнути проблеми затухання градієнтів в глибоких мережах.


In [18]:
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, kernel_size=3, dropout=0.1):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv1d(in_channels, in_channels, kernel_size,
                              padding=kernel_size // 2)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.LeakyReLU(negative_slope=0.1)

    def forward(self, x):
        out = self.conv(x)
        out = self.dropout(out)
        out = self.activation(out)
        return out

class ConvResNet(nn.Module):
    def __init__(self, in_channels, num_layers, kernel_size=3, dropout=0.1):
        super(ConvResNet, self).__init__()
        self.layers = nn.ModuleList([
            ConvBlock(in_channels, kernel_size, dropout) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            x = x + out
        return x

Я створила клас TokenEmbedding, який реалізує шар для векторного представлення токенів:
1.	nn.Embedding — використовується для перетворення індексів токенів у їх відповідні вектори в заданому просторі розмірності (embedding_dim).
2.	forward — функція, що приймає тензор з індексами токенів, перетворює його на вектори, а потім змінює розміри тензора, щоб відповідати формату для подальшої обробки.
Цей клас створює векторні представлення для токенів у послідовностях, готові до використання в нейронній мережі.


In [19]:
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, x):
        batch_size, max_sentence_len, max_token_len = x.size()
        # "Сплющуємо" батч та речення в один вимір для обробки кожного слова окремо
        x = x.view(-1, max_token_len)
        x = self.embedding(x)
        # Змінюємо порядок вимірів для Conv1d: (batch, channels, length)
        x = x.permute(0, 2, 1)
        return x

Я створила клас TokenPOSTagger, який є моделлю для виявлення частин мови (POS) на основі вбудовувань токенів та згорткових мереж:
1.	TokenEmbedding — створює векторні представлення для кожного токена в послідовності.
2.	ConvResNet — згорткова мережа з резидуальними з'єднаннями для обробки векторів токенів.
3.	AdaptiveMaxPool1d — глобальне зниження вимірів за допомогою максимального пулінгу по всій довжині послідовності.
4.	nn.Linear — лінійний шар, що перетворює отримані ознаки в логіти для кожного POS-тегу.
Функція forward обробляє вхідні токени через ці компоненти та повертає логіти для кожного токена в реченні, що відповідають за виявлення POS-тегів.


In [20]:
import torch
import torch.nn as nn

class TokenPOSTagger(nn.Module):
    def __init__(self, vocab_size, labels_num, embedding_size=32,
                 num_res_layers=3, kernel_size=3, dropout=0.1):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, embedding_size)
        self.backbone = ConvResNet(
            in_channels=embedding_size,
            num_layers=num_res_layers,
            kernel_size=kernel_size,
            dropout=dropout
        )
        self.global_pooling = nn.AdaptiveMaxPool1d(1)
        self.out = nn.Linear(embedding_size, labels_num)

    def forward(self, tokens):
        batch_size, max_sentence_len, max_token_len = tokens.shape
        # Отримуємо ембеддінги символів
        char_embeddings = self.token_embedding(tokens)
        # Проганяємо через згорткові шари (ResNet)
        features = self.backbone(char_embeddings)
        # Вибираємо найбільш значущі ознаки для кожного слова
        global_features = self.global_pooling(features).squeeze(-1)
        # Фінальна класифікація
        logits_flat = self.out(global_features)
        # Повертаємо структуру до вигляду (batch, sent_len, labels_num)
        logits = logits_flat.view(batch_size, max_sentence_len, -1)
        return logits

Я створила модель TokenPOSTagger з наступними параметрами:
•	vocab_size = 100 (розмір словника токенів).
•	labels_num = 17 (кількість класів для POS-тегів).
•	embedding_size = 32 (розмірність вбудовувань токенів).
•	num_res_layers = 3 (кількість згорткових блоків в мережі).
•	kernel_size = 3 (розмір ядра згортки).
•	dropout = 0.1 (ймовірність відключення нейронів під час тренування).
Модель була протестована на випадкових токенах розміру (2, 5, 10) і отримано тензор з логітами для кожного токена.
Виведено форму тензора logits, що є результатом роботи моделі: (2, 5, 17), що означає, що модель вивела 17 логітів для кожного з 5 токенів у 2 реченнях.


In [21]:
vocab_size = 100
labels_num = 17
embedding_size = 32
num_res_layers = 3
kernel_size = 3
dropout = 0.1

model = TokenPOSTagger(vocab_size, labels_num, embedding_size,
                       num_res_layers, kernel_size, dropout)

# Генеруємо випадковий тензор: (batch_size=2, sent_len=5, token_len=10)
tokens = torch.randint(0, vocab_size, (2, 5, 10))
logits = model(tokens)
print(logits.shape)

torch.Size([2, 5, 17])


Я створила функцію train_eval_loop, яка виконує цикл тренування та валідації моделі:
1.	optimizer — ініціалізує оптимізатор, використовуючи Adam.
2.	lr_scheduler — налаштовує адаптивний графік навчання, який зменшує швидкість навчання, якщо валідаційна втрата не покращується.
3.	Для кожної епохи:
o	Модель тренується на даних (оновлюються ваги на основі втрат).
o	Виконується валідація без оновлення ваг.
4.	Функція save_model зберігає модель, якщо валідаційна втрата зменшується.
5.	Після завершення циклу функція повертає списки втрат для тренування та валідації.


In [22]:
def train_eval_loop(model, train_loader, val_loader, num_epochs=5,
                    lr=0.01, lr_decay=0.5, patience=2,
                    loss_fn=nn.CrossEntropyLoss(),
                    optimizer_ctor=torch.optim.Adam, device="cpu",
                    save_path="best_model_4_1_1.pth"):

    optimizer = optimizer_ctor(model.parameters(), lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=patience, factor=lr_decay
    )

    train_losses, val_losses = [], []
    best_val_loss = float("inf")

    for epoch in range(1, num_epochs + 1):
        # Режим навчання
        model.train()
        total_train_loss = 0
        for batch in train_loader:
            sentences, labels = batch
            sentences, labels = sentences.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(sentences)

            # Сплющуємо вихід для CrossEntropyLoss: (batch * sent_len, classes)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        train_loss = total_train_loss / len(train_loader)
        train_losses.append(train_loss)

        # Режим валідації
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                sentences, labels = batch
                sentences, labels = sentences.to(device), labels.to(device)
                logits = model(sentences)
                loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
                total_val_loss += loss.item()

        val_loss = total_val_loss / len(val_loader)
        val_losses.append(val_loss)

        print(f"Epoch {epoch}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        # Збереження найкращої моделі
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_model(model, save_path)

        # Зменшення швидкості навчання (Learning Rate), якщо прогрес зупинився
        lr_scheduler.step(val_loss)

    return train_losses, val_losses

Я створила функцію evaluate_model, яка оцінює продуктивність моделі на вхідних даних:
1.	model.eval() — переводить модель у режим оцінки, щоб вимкнути деякі операції (наприклад, дроп-аут).
2.	Для кожного батчу з data_loader:
o	Виконується передбачення за допомогою моделі.
o	Визначаються індекси найбільших логітів для передбачених класів.
3.	classification_report з бібліотеки sklearn обчислює основні метрики, такі як точність, відчутність та F1-міра для кожного класу.
4.	Функція виключає мітки з класом <PAD>, замінюючи їх на <UNK> (не відомий клас).


In [23]:
from sklearn.metrics import classification_report
import numpy as np

def evaluate_model(model, data_loader, label_vocab, device="cpu"):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in data_loader:
            sentences, labels = batch
            sentences, labels = sentences.to(device), labels.to(device)

            logits = model(sentences)
            # Вибираємо індекс з найбільшою ймовірністю для кожного слова
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy().reshape(-1))
            all_labels.extend(labels.cpu().numpy().reshape(-1))

    # Створюємо зворотний словник: індекс -> назва частини мови
    id_to_label = {idx: label for label, idx in label_vocab.items()}

    filtered_preds = []
    filtered_labels = []

    # Важливо: ігноруємо токен падінгу (label != 0) при розрахунку метрик
    for pred, label in zip(all_preds, all_labels):
        if label != 0:
            filtered_preds.append(id_to_label.get(pred, "<UNK>"))
            filtered_labels.append(id_to_label.get(label, "<UNK>"))

    if not filtered_preds or not filtered_labels:
        print("Помилка: списки передбачень або міток порожні!")
        return

    # Генеруємо звіт: Precision, Recall, F1-score для кожної категорії
    report = classification_report(filtered_labels, filtered_preds, output_dict=False)
    print(report)

Я налаштувала модель TokenPOSTagger для POS-тегування, використовуючи вбудовування та згорткові шари. Створив підмножини тренувальних і валідаційних даних, завантажуючи їх через DataLoader з батчами по 8 прикладів. Навчаю модель на 25 епох з адаптивним зменшенням швидкості навчання, оцінюючи її на тренувальних і валідаційних наборах за допомогою функції evaluate_model.

In [25]:
vocab_size = len(char_vocab)
labels_num = len(pos_vocab)
embedding_size = 64
num_res_layers = 3
kernel_size = 3
dropout = 0.3

model = TokenPOSTagger(vocab_size, labels_num, embedding_size,
                       num_res_layers, kernel_size, dropout)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

train_dataset = TensorDataset(train_char_tensor, train_pos_tensor)
val_dataset = TensorDataset(test_char_tensor, test_pos_tensor)

from torch.utils.data import DataLoader, Subset
batch_size = 8
subset_size = 1000

train_subset = Subset(train_dataset, range(subset_size))
val_subset = Subset(val_dataset, range(subset_size // 2))

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

train_eval_loop(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=5,
    lr=5e-3,
    lr_decay=0.5,
    patience=5,
    loss_fn=torch.nn.CrossEntropyLoss(),
    optimizer_ctor=torch.optim.Adam,
    device=device
)

print("Оцінка на тренувальному датасеті:")
evaluate_model(model, train_loader, pos_vocab, device)
print("Оцінка на валідаційному датасеті:")
evaluate_model(model, val_loader, pos_vocab, device)

Epoch 1/5 - Train Loss: 0.2257, Val Loss: 0.1154
Модель збережено в best_model_4_1_1.pth
Epoch 2/5 - Train Loss: 0.0765, Val Loss: 0.0806
Модель збережено в best_model_4_1_1.pth
Epoch 3/5 - Train Loss: 0.0606, Val Loss: 0.0722
Модель збережено в best_model_4_1_1.pth
Epoch 4/5 - Train Loss: 0.0525, Val Loss: 0.0684
Модель збережено в best_model_4_1_1.pth
Epoch 5/5 - Train Loss: 0.0477, Val Loss: 0.0599
Модель збережено в best_model_4_1_1.pth
Оцінка на тренувальному датасеті:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

         ADJ       0.71      0.91      0.80      1490
         ADP       0.94      0.98      0.96      1391
         ADV       0.81      0.70      0.75      1123
         AUX       0.73      0.99      0.84       192
       CCONJ       0.88      0.89      0.89       741
         DET       0.82      0.82      0.82       720
        INTJ       0.00      0.00      0.00        25
        NOUN       0.87      0.82      0.84      3525
         NUM       1.00      0.29      0.45        62
        PART       0.80      0.83      0.81       750
        PRON       0.93      0.82      0.87       873
       PROPN       0.81      0.69      0.75       414
       PUNCT       1.00      1.00      1.00      3331
       SCONJ       0.87      0.86      0.86       408
         SYM       1.00      0.70      0.82        10
        VERB       0.89      0.90      0.90      2093
           X       0.98      0.75      0.85        53

    accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

***Контрольні питання***



1.	Що таке POS-тегування? Чому ця задача важлива для обробки тексту?
Відповідь: POS-тегування (Part-Of-Speech tagging) — це процес визначення граматичної категорії (частини мови) для кожного слова в тексті. Наприклад, визначення, чи є слово іменником, дієсловом, прикметником тощо. Ця задача важлива для обробки тексту, оскільки допомагає зрозуміти структуру речення і контекст, що необхідно для виконання багатьох задач в обробці природної мови, таких як машинний переклад, аналіз сентименту, розпізнавання сутностей тощо.

2.	Які кроки необхідно виконати для підготовки корпусу до навчання моделі?
Відповідь: Підготовка корпусу до навчання моделі включає кілька основних етапів:
•	Збір та очищення даних: видалення зайвих символів, нормалізація тексту (наприклад, приведення до нижнього регістру).
•	Токенізація: розбиття тексту на токени (слова, символи або підслівні одиниці).
•	Лемматизація або стемінг: приведення слів до їх базових форм.
•	Тегування (якщо необхідно): маркування частин мови для задач типу POS-тегування.
•	Розподіл на навчальну та тестову вибірки: для подальшого оцінювання моделі.

3.	Як модель враховує локальні патерни символів у токенах?
Відповідь: Модель може враховувати локальні патерни символів у токенах через використання різних архітектур, таких як згорткові нейронні мережі (CNN), які здатні вивчати патерни на рівні символів у словах. Це дозволяє моделі враховувати, наприклад, суфікси чи префікси слів, що допомагає в розпізнаванні їх значення навіть за умов омонімії чи морфологічних варіантів.

4.	Чому ми додаємо <PAD> токени до символів і речень?
Відповідь: <PAD> токени додаються для того, щоб усі входи в модель мали однакову довжину. Це важливо для пакетної обробки даних, оскільки нейронні мережі працюють з фіксованими розмірами вхідних даних. Таким чином, заповнення коротших текстів або токенів <PAD> дозволяє створити однакові розміри для всіх вхідних даних у модель.

5.	Які метрики використовуються для оцінки якості моделі? Як вони інтерпретуються?
Відповідь: Для оцінки якості моделі в задачах NLP використовуються різні метрики:
•	Точність (Accuracy): частка правильних передбачень серед усіх прикладів. Інтерпретується як процент правильних передбачень.
•	Точність (Precision): частка правильних позитивних передбачень серед усіх передбачених позитивних випадків. Важлива для задач, де важливо уникати хибних позитивних результатів.
•	Повнота (Recall): частка правильних позитивних передбачень серед усіх реальних позитивних випадків. Важлива для задач, де важливо виявити всі позитивні приклади.
•	F1-міра: гармонійне середнє точності та повноти. Використовується, коли важливий баланс між цими двома метриками.

6.	Що таке ResNet-з'єднання, і як вони допомагають у навчанні глибоких моделей?
Відповідь: ResNet-з'єднання (Residual connections) — це специфічна архітектурна особливість глибоких нейронних мереж, яка дозволяє передавати вихід попереднього шару без змін через прямий зв'язок до наступного шару. Це допомагає вирішити проблему затухаючих градієнтів в глибоких мережах, що дозволяє моделям вчитися швидше та більш ефективно, зберігаючи стабільність навчання.

7.	Як працює шар згорток у моделі? Які параметри впливають на його роботу?
Відповідь: Шар згорток (Convolutional Layer) застосовує фільтри (ядра згортки) до вхідних даних для виявлення локальних патернів (наприклад, ліній, кутів, текстур) в зображеннях або текстах. Параметри, які впливають на його роботу, включають:
•	Розмір фільтра: визначає, скільки сусідніх елементів буде охоплено фільтром.
•	Крок (Stride): визначає, наскільки далеко фільтр переміщається після кожного застосування.
•	Заповнення (Padding): додається для збереження розмірів вхідних даних після застосування фільтра.
•	Кількість фільтрів: визначає кількість різних ознак, які модель може навчити.

8.	Яка роль шару AdaptiveMaxPool1d у моделі?
Відповідь: Шар AdaptiveMaxPool1d виконує операцію максимального пулінгу на вхідних даних. Він адаптує вихідний розмір так, щоб він був однаковим, незалежно від розміру вхідного сигналу. Це корисно для зменшення розміру даних, одночасно зберігаючи найбільш важливі ознаки, що підвищує ефективність моделі.

9.	Як модель перетворює символи у багатовимірні ембеддинги? Чому це важливо?
Відповідь: Модель перетворює символи в багатовимірні ембеддинги за допомогою векторних представлень, таких як Word2Vec, GloVe або використовуючи навчання через нейронні мережі. Кожен символ або слово відображається як вектор, що дозволяє враховувати контекст і значення слова в багатовимірному просторі. Це важливо, оскільки багатовимірні ембеддинги дозволяють моделі краще розуміти семантичні зв'язки між словами.

10.	Чому модель, яка працює тільки на рівні символів, може помилятися у випадках омонімії або складних синтаксичних конструкцій?
Відповідь: Моделі, які працюють тільки на рівні символів, часто не враховують контекстуальну залежність, яка є важливою для розуміння омонімії та складних синтаксичних конструкцій. Омонімія виникає, коли одне слово має кілька значень, і без контексту модель не може точно визначити правильне значення. Складні синтаксичні конструкції також можуть бути важкими для моделі, оскільки вона не завжди розпізнає, як слова взаємодіють між собою в межах великого контексту.

